# 03 · Live win probability
**Question:** at minute *t* of a match, how likely is the Hidden King side to win? Drawn as a curve on the match page.
**Inputs (team 0 minus team 1, at each 3-minute snapshot):** net worth, kills, deaths, assists, level, plus the minute and net worth share.
**Leakage rules:** no match duration, no final stats, and the extra snapshot a second before the end is dropped.
**Split:** by match time; every snapshot of a match is on the same side. **Report:** overall metrics and AUC per minute.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from broadway_ml.flatten import load_tables, item_catalog
from broadway_ml import features as fe, evaluate as ev, export as ex
T = load_tables('../data/tables')
{k: len(v) for k, v in T.items()}

In [ ]:
tl = fe.timeline_features(T['matches'], T['stats'])
start = tl.match_id.map(T['matches'].set_index('match_id').start_time).to_numpy()
tr, te = ev.time_split(start)
FEATS = ['net_worth_diff', 'kills_diff', 'deaths_diff', 'assists_diff', 'level_diff', 'net_worth_share', 'minute']
tl.groupby('minute').size().head(15)

## Baseline
Net worth lead alone, scaled by game time: a one-feature logistic regression on `net_worth_share`. Hard to beat early on, so it is the bar.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
b = LogisticRegression().fit(tl.loc[tr, ['net_worth_share']], tl.loc[tr, 'win'])
preds = {'base rate': np.full(te.sum(), tl.loc[tr, 'win'].mean()),
         'net worth share': b.predict_proba(tl.loc[te, ['net_worth_share']])[:, 1]}
ev.compare(tl.loc[te, 'win'], preds)

## Your model
A lead of 5k souls means much more at minute 30 than at minute 6, so the model needs time interactions. Options that still export as `linear@1`:
- add products such as `net_worth_diff * minute` as extra columns (the site computes the same products)
- or one small logistic model per time bucket (0-9, 9-18, 18-27, 27+ min) exported as separate files
- objectives (Walkers, Guardians, Patron phases) are in the API (`include_objectives`) and are likely the strongest late-game signal: worth pulling once the basics work

In [ ]:
# TODO(V): feature engineering and the model
tl['nw_x_min'] = tl.net_worth_diff * tl.minute
F = FEATS + ['nw_x_min']
scaler = StandardScaler().fit(tl.loc[tr, F])
model = LogisticRegression(C=1.0, max_iter=2000).fit(scaler.transform(tl.loc[tr, F]), tl.loc[tr, 'win'])
preds['logistic'] = model.predict_proba(scaler.transform(tl.loc[te, F]))[:, 1]
report = ev.compare(tl.loc[te, 'win'], preds); report

In [ ]:
per_min = pd.concat({k: ev.auc_by_minute(tl[te], v).set_index('minute').auc for k, v in preds.items() if k != 'base rate'}, axis=1)
per_min.plot(title='AUC by minute', ylim=(0.5, 1.0));

In [ ]:
ev.plot_reliability(tl.loc[te, 'win'], {k: v for k, v in preds.items() if k != 'base rate'});

In [ ]:
ex.export_linear(model, F, '../models/live_win.json', scaler=scaler, name='live-win', metrics=report.loc['logistic'].to_dict(),
                 notes='features are team 0 minus team 1 at a 3-minute snapshot; nw_x_min = net_worth_diff * minute')